# Error Case 8 — 409 Conflict: Namespace Not Empty

**Error:** `409 Conflict`  
**Root cause:** Deleting namespace that still contains tables or child namespaces  
**Fix:** Delete bottom-up: tables → child namespaces → parent namespace

## Flow
```
DELETE /catalog/v1/{catalog}/namespaces/{ns}
  → check if empty
  → has child tables/namespaces → 409 NamespaceNotEmptyException
```

In [14]:
import sys
sys.path.insert(0, '..')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [15]:
# Cell 2: Setup — catalog + namespace + child namespace
r1 = create_catalog()
assert r1.status_code == 201, r1.text

r2 = create_namespace()
assert r2.status_code in [200, 201, 500], r2.text

# Create child namespace inside parent
r3 = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(),
    json={'namespace': [TEST_NAMESPACE, 'child'], 'properties': {}}
)
assert r3.status_code in [200, 201, 500], r3.text

print('Setup: catalog ✅ | namespace ✅ | child namespace ✅')

Setup: catalog ✅ | namespace ✅ | child namespace ✅


In [16]:
# Cell 3: Trigger 409 — delete parent namespace (has child)
r = requests.delete(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}',
    headers=h()
)
print_response(r, '409 Namespace not empty')
assert r.status_code == 409
print('\n✅ 409 confirmed — namespace not empty')


409 Namespace not empty:
  Status:     409
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000001034
  Body: {
  "error": {
    "message": "Namespace test-ns is not empty",
    "type": "NamespaceNotEmptyException",
    "code": 409
  }
}

✅ 409 confirmed — namespace not empty


In [17]:
# Cell 4: Find log in OpenSearch
wait_for_logs(5)
request_id = get_request_id(r)
hits = search_409_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Found 10 log entries:

ℹ️  [2026-06-02T01:57:59.369Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000001034
   realmId   : POLARIS
   message   : Handling runtimeException Namespace test-ns is not empty

ℹ️  [2026-06-02T01:57:59.369Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000001034
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:01:57:59 +0000] "DELETE /api/catalog/v1/test-error-catalog/namespaces/test-ns HTTP/1.1" 409 101

ℹ️  [2026-06-02T01:57:53.895Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000001025
   realmId   : POLARIS
   message   : Handling runtimeException TopLevelEntity of type PRINCIPAL_ROLE does not exist: test-principal-role

ℹ️  [2026-06

In [18]:
# Cell 5: Fix — delete bottom-up
# Step 1: Delete child namespace first
r_child = requests.delete(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}%1Fchild',
    headers=h()
)
print(f'Delete child namespace: {r_child.status_code}')

# Step 2: Now delete parent (empty)
r_parent = requests.delete(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}',
    headers=h()
)
print(f'Delete parent namespace: {r_parent.status_code}')
assert r_parent.status_code == 204
print('\n✅ Fixed — deleted bottom-up')

cleanup()
print('\n=== Summary ===')
print('Error:    409 Conflict')
print('Cause:    Namespace contains child namespaces or tables')
print('Detect:   ERROR message contains empty/NamespaceNotEmpty')
print('Fix:      Delete bottom-up: child namespaces → tables → parent')
print('Note:     Nested namespace separator in URL: parent%1Fchild')

Delete child namespace: 204
Delete parent namespace: 204

✅ Fixed — deleted bottom-up
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    409 Conflict
Cause:    Namespace contains child namespaces or tables
Detect:   ERROR message contains empty/NamespaceNotEmpty
Fix:      Delete bottom-up: child namespaces → tables → parent
Note:     Nested namespace separator in URL: parent%1Fchild
